# 5. The configuration model: from neighbour support to a prediction

The triangle model builds connections in groups of three. To understand what this
structure changes, we compare it with a graph that has the same **limiting degree
distribution**, but pairs its edges without forming prescribed communities.

This notebook derives the configuration-model prediction for the fraction of
vertices in the 3-core. It also defines the numerical functions used in
`06_configuration_simulation.ipynb`, which runs the simulations, and
`07_configuration_analysis.ipynb`, which draws the comparison figures.
Loading this notebook only defines functions; it does not
simulate graphs or write files.

The main distinction is between two parameters:

- $\lambda$ is the mean number of triangle memberships per individual.
- $\mu=2\lambda$ is the limiting mean **graph degree** in both models.

Keeping these separate matters: this comparison is not with an Erdős–Rényi
graph of Poisson degree. Our configuration-model degree is twice a Poisson
random variable.

## Which degree distribution should we match?

An individual with $D$ triangle memberships usually meets two different
neighbours in each membership. In a finite graph, repeated individuals within
a community or overlaps between communities can reduce its number of distinct
neighbours. With fixed $\lambda$ and growing $n$, these coincidences affect
a vanishing fraction of vertices. The limiting degree is therefore

$$
K=2D,\qquad D\sim\operatorname{Poisson}(\lambda),\qquad
\mu=\mathbb{E}[K]=2\lambda.
$$

Its probability mass function is

$$
\pi_{2j}=e^{-\lambda}\frac{\lambda^j}{j!},\qquad \pi_{2j+1}=0.
$$

Matching only the mean would miss the even-valued distribution and its
variance, $\operatorname{Var}(K)=4\lambda$.

This specialization agrees with equation (2.20) and Corollary 2.9 of
[van der Hofstad, Komjáthy and Vadon, *Random intersection graphs with
communities*](https://arxiv.org/pdf/1809.02514#page=9):
the limiting degree is the sum of the internal degrees contributed by all
memberships. For a triangle, every internal degree is two. Their local limit
retains communities; it is not an ordinary tree.

Our triangle sampler conditions the whole membership vector on its sum being
divisible by three. At fixed positive $\lambda$, this global constraint does
not change the limiting degree law. The comparison matches that law, rather
than promising identical degree sequences in each pair of finite graphs.

## Why an edge can be studied through neighbour support

To construct a configuration model with degrees $K_1,\ldots,K_n$, give vertex
$i$ exactly $K_i$ half-edges and pair all half-edges uniformly. The sum must
be even; our degrees are already even. A pair can form a self-loop, and several
pairs can connect the same two vertices.

For fixed $\lambda$, exploring a bounded neighbourhood is unlikely to visit
the same vertex twice as $n$ grows. The limiting neighbourhood is a branching
tree, so support arriving along different branches becomes independent. This is
the reason a scalar message equation works here. It would fail for the triangle
graph itself: two neighbours in one triangle are connected and can support one
another.

Imagine temporarily removing one edge from a vertex. The vertex sends a
**support message** along that edge if at least two of its other neighbours
can support it. If the removed neighbour also supports the vertex, the vertex
then has the three neighbours required for the 3-core.

Let $x$ be the probability of such a message. Following a uniformly chosen
edge does not reach a uniformly chosen vertex. A degree-$k$ vertex is $k$
times as likely to be reached, giving the size-biased distribution

$$
\mathbb{P}(K^*=k)=\frac{k\pi_k}{\mu}.
$$

The reached vertex has $k-1$ other neighbours. In the limiting tree, the number
that support it is $\operatorname{Binomial}(k-1,x)$. Therefore

$$
\boxed{\quad
x=F(x)=\sum_{k\ge1}\frac{k\pi_k}{\mu}
       \mathbb{P}\!\left(\operatorname{Binomial}(k-1,x)\ge2\right).
\quad}
$$

For a uniformly chosen vertex, all $k$ neighbours are available. It remains
in the core when at least three support it, so the predicted density is

$$
\boxed{\quad
\rho(x)=\sum_{k\ge0}\pi_k
       \mathbb{P}\!\left(\operatorname{Binomial}(k,x)\ge3\right).
\quad}
$$

There may be several fixed points, including $x=0$. The relevant one is the
**largest** fixed point. One way to understand this is to start with every
message present ($x_0=1$) and repeatedly remove messages that lack support.
The decreasing sequence $x_{t+1}=F(x_t)$ reaches the largest fixed point.
Starting at zero would stay at zero and miss an existing core.

## How this connects to the cited theorem

[Janson and Łuczak, *A simple solution to the $k$-core problem*,
Condition 2.2 and Theorem 2.3](https://arxiv.org/pdf/math/0508453#page=4)
use a thinned degree $K_x\mid K\sim\operatorname{Binomial}(K,x)$.
For the 3-core, write

$$
h(x)=\mathbb{E}[K_x\,\mathbf 1_{\{K_x\ge3\}}],
\qquad h_1(x)=\mathbb{P}(K_x\ge3).
$$

The binomial identity

$$
\mathbb{E}[\operatorname{Binomial}(k,x)\,
             \mathbf 1_{\{\operatorname{Binomial}(k,x)\ge3\}}]
=kx\,\mathbb{P}(\operatorname{Binomial}(k-1,x)\ge2)
$$

gives $h(x)=\mu xF(x)$. Thus their equation $\mu x^2=h(x)$
is exactly our support equation for $x>0$, and $h_1(x)=\rho(x)$.

The theorem assumes convergence of the empirical degree distribution and its
mean. These hold for our independent twice-Poisson degrees; all their moments
are finite. When the largest positive root has $h(x)>\mu x^2$ just below it,
the multigraph core fraction converges in probability to $h_1(x)$.
When no positive root exists, the limiting fraction is zero.
The tangency at the critical parameter does **not** satisfy that crossing
condition, so this statement does not assign a deterministic core fraction
at criticality.

Our simulation retains the **CM multigraph**, counting loops twice in degree
and retaining parallel edges. This is the direct comparison with Theorem 2.3.
It also records a **CM erased** control: the very same pairing after removing
loops and merging repeated edges, so degree counts distinct neighbours as in
the triangle graph. These two observations are paired, not independent.

Theorem 2.4 treats a uniformly sampled simple graph with prescribed degrees
under additional moment conditions, satisfied here. Erasing a multigraph is
a different finite-$n$ distribution from conditioning on simplicity. Defects
are sparse at fixed $\lambda$, but deleting even a few edges can cause a
pruning cascade near the transition. Notebook 06 records these defects, and
notebook 07 measures their effects. The theorem does not provide an exact
formula for the erased control or for any particular finite graph.

## Work out the formulas for twice-Poisson degrees

Probability generating functions make the binomial sums short. Define

$$
G_0(z)=\mathbb{E}[z^K]=e^{\lambda(z^2-1)},\qquad
G_1(z)=\mathbb{E}[z^{K^*-1}]
      =\frac{G_0'(z)}{\mu}=z\,e^{\lambda(z^2-1)}
$$

for $\lambda>0$. Equivalently, $K^*-1$ has distribution
$1+2\operatorname{Poisson}(\lambda)$. The extra one is an ordinary
remaining half-edge; it is not a prescribed triangle connection.

For any nonnegative integer $L$ with generating function $G$,
the probabilities of zero and one supporting neighbour are $G(1-x)$
and $xG'(1-x)$. Applying this to $L=K^*-1$ gives

$$
F_\lambda(x)=1-G_1(1-x)-xG_1'(1-x).
$$

Since $G_1'(z)=e^{\lambda(z^2-1)}(1+2\lambda z^2)$, this simplifies to

$$
\boxed{
F_\lambda(x)=1-e^{-\lambda x(2-x)}
                    \bigl[1+2\lambda x(1-x)^2\bigr].
}
$$

For a vertex to remain in the 3-core, exclude zero, one, and two supporting
neighbours:

$$
\rho_\lambda(x)=1-G_0(1-x)-xG_0'(1-x)
                    -\frac{x^2}{2}G_0''(1-x).
$$

Using $G_0'(z)=2\lambda zG_0(z)$ and
$G_0''(z)=(2\lambda+4\lambda^2z^2)G_0(z)$, we obtain

$$
\boxed{
\rho_\lambda(x)=1-e^{-\lambda x(2-x)}
 \bigl[1+2\lambda x(1-x)+\lambda x^2
                 +2\lambda^2x^2(1-x)^2\bigr].
}
$$

These are our specialization of the general equations, rather than formulas
quoted directly from the paper. At $\lambda=0$, the graph has no edges and
the predicted density is zero. The code also returns zero for $F_0$:
there is then no edge along which to define a size-biased vertex.

In [ ]:
import numpy as np


def _cm_inputs(x, lam):
    """Check the probability x and the nonnegative membership mean lambda."""
    x = np.asarray(x, dtype=float)
    lam = float(lam)
    if not np.isfinite(lam) or lam < 0:
        raise ValueError("lambda must be finite and nonnegative.")
    if not np.all(np.isfinite(x)) or np.any((x < 0) | (x > 1)):
        raise ValueError("The support probability x must lie in [0, 1].")
    return x, lam


def cm_message(x, lam):
    """Probability of at least two other supports; x may be a NumPy array."""
    x, lam = _cm_inputs(x, lam)
    exponent = lam * x * (2 - x)
    # expm1 evaluates exp(t) - 1 accurately when t is close to zero.
    result = -np.expm1(-exponent) - (2 * lam * x * (1 - x) ** 2 * np.exp(-exponent))
    result = np.clip(result, 0.0, 1.0)
    return float(result) if result.ndim == 0 else result


def cm_density(x, lam):
    """Fraction with at least three supports at a specified support probability."""
    x, lam = _cm_inputs(x, lam)
    exponent = lam * x * (2 - x)
    one_or_two = 2 * lam * x * (1 - x) + lam * x**2 + 2 * lam**2 * x**2 * (1 - x) ** 2
    result = -np.expm1(-exponent) - one_or_two * np.exp(-exponent)
    result = np.clip(result, 0.0, 1.0)
    return float(result) if result.ndim == 0 else result


def cm_message_derivative(x, lam):
    """Derivative of F with respect to x, used to locate its tangency."""
    x, lam = _cm_inputs(x, lam)
    result = (
        2
        * lam
        * x
        * (1 - x)
        * np.exp(-lam * x * (2 - x))
        * (3 + 2 * lam * (1 - x) ** 2)
    )
    return float(result) if result.ndim == 0 else result

## Locate the transition without missing a narrow branch

For every finite $\lambda$, $F_\lambda(0)=0$ and
$F_\lambda'(0)=0$. In fact,

$$
F_\lambda(x)=(3\lambda+2\lambda^2)x^2+O(x^3).
$$

A positive root therefore does not grow continuously from the zero root at a
finite parameter. Instead, the first positive roots appear when the curve
$F_\lambda(x)$ touches the line $x$ at an interior point:

$$
F_{\lambda_c}(x_c)=x_c,\qquad
F_{\lambda_c}'(x_c)=1.
$$

The derivative needed below is

$$
F_\lambda'(x)=2\lambda x(1-x)e^{-\lambda x(2-x)}
                        \bigl[3+2\lambda(1-x)^2\bigr].
$$

Here is a convenient way to find that touching point.

1. For a proposed $0<x<1$, solve $F_\lambda(x)=x$ for $\lambda$.
   The solution is unique because $F_\lambda(x)$ increases strictly with
   $\lambda$, from zero to one. Call it $\Lambda(x)$.
2. Find the minimum of $\Lambda(x)$. It diverges near both ends of $(0,1)$,
   so its minimum is in the interior.
3. Refine the bracket by solving the tangency condition. Implicit
   differentiation gives

   $$
   \Lambda'(x)=\frac{1-F_{\Lambda(x)}'(x)}
                    {\partial_\lambda F_\lambda(x)|_{\lambda=\Lambda(x)}}.
   $$

   Along this branch, the message derivative $F_{\Lambda(x)}'(x)$ changes
   from above one to below one across the minimum; $\Lambda'(x)$ changes
   from negative to positive.

The initial grid below only brackets the minimum. Both the degree parameter and
the tangency are then found by bisection on continuous functions. We do not use
the grid as the threshold estimate or hope that it hits two closely spaced
roots.

The result is approximately

$$
\lambda_c=1.28772037,\qquad x_c=0.43903221,\qquad
\rho_c=\rho_{\lambda_c}(x_c)=0.13788881.
$$

The limiting prediction thus jumps from zero below the threshold to a positive
right-hand limit of about $13.8\%$. The corresponding mean degree is
$2\lambda_c\approx2.57544$. This is a prediction for twice-Poisson degrees,
not the familiar Poisson-degree Erdős–Rényi threshold.

## Why this is the first and only critical touching point

A numerical minimum is useful, but the configuration-model fluctuation theorem
also asks for a unique nondegenerate interior minimum. For our degree family,
we can verify the corresponding shape analytically near the first transition.

Fix $0<\lambda\leq1.3$. The derivative $F_\lambda'(x)$ is positive on $(0,1)$
and vanishes at both endpoints. Its logarithm has second derivative

$$
\frac{d^2}{dx^2}\log F_\lambda'(x)
=-\frac{1}{x^2}-\frac{1}{(1-x)^2}+2\lambda
+\frac{4\lambda[3-2\lambda(1-x)^2]}
       {[3+2\lambda(1-x)^2]^2}
\leq -8+\frac{10}{3}\lambda<0.
$$

Here the first two terms sum to at most $-8$, and the final fraction is at most
$4\lambda/3$. Thus the logarithm is strictly concave: its slope decreases
strictly. In particular, $F_\lambda'$ rises to one peak and then falls.
It can cross the level one at most twice.

For $G(x,\lambda)=F_\lambda(x)-x$, this means that its slope can have the signs
minus, plus, minus, with at most one interior local maximum. We also have
$G(0,\lambda)=0$ and $G(1,\lambda)=-e^{-\lambda}<0$. If
$\lambda\leq1/2$, the bound

$$
F_\lambda'(x)\leq\frac{\lambda(3+2\lambda)}{2}\leq1
$$

rules out a positive zero; the first inequality is strict where needed.
At $\lambda=1.3$, on the other hand, $F_{1.3}(0.44)>0.44$.
This last sign does not depend on a delicate numerical comparison: putting
$t=0.89232$ gives
$e^t>\sum_{j=0}^4t^j/j!>2.4352$,
whereas $(1+2(1.3)(0.44)(0.56)^2)/0.56<2.4264$.

For each fixed $0<x<1$, $G$ increases strictly with $\lambda$. The first
positive zero therefore occurs for $1/2<\lambda_c<1.3$, when the unique
interior maximum just reaches zero. The maximizing coordinate stays away
from the endpoints: $F_\lambda(x)=O(x^2)$ uniformly near zero in this bounded
parameter interval, and $G(1,\lambda)$ stays strictly negative.
At that point $F_\lambda'$ crosses one on its descending side, so
$G_{xx}<0$. This proves that the critical touching point is unique and
nondegenerate, rather than merely suggesting it from a plot.

In the theorem's notation,
$\ell(x)=\mu x^2-h(x)=-2\lambda xG(x,\lambda)$.
At criticality it is nonnegative, has exactly one zero in $(0,1]$, and has
$\ell''(x_c)=-2\lambda_c x_cG_{xx}(x_c,\lambda_c)>0$.
This verifies the shape condition; degree-sequence assumptions and stochastic
limits must still be checked separately.

In [ ]:
def _cm_bisect(function, left, right, tolerance=1e-15):
    """Solve a sign-bracketed scalar equation by repeatedly halving the interval."""
    left_value = function(left)
    right_value = function(right)
    if left_value == 0:
        return float(left)
    if right_value == 0:
        return float(right)
    if left_value * right_value > 0:
        raise ValueError("Bisection needs opposite signs at the interval ends.")
    for _ in range(120):
        middle = (left + right) / 2
        middle_value = function(middle)
        if middle_value == 0 or right - left <= tolerance * max(1, abs(middle)):
            return float(middle)
        if left_value * middle_value > 0:
            left, left_value = middle, middle_value
        else:
            right = middle
    raise RuntimeError("Bisection did not reach the requested accuracy.")


def _cm_lambda_for_message(x):
    """Find the unique lambda for which a chosen positive x is a fixed point."""
    if not 0 < x < 1:
        raise ValueError("A positive fixed-point branch requires 0 < x < 1.")
    upper = 1.0
    while cm_message(x, upper) < x:
        upper *= 2
    return _cm_bisect(lambda lam: cm_message(x, lam) - x, 0.0, upper)


def cm_critical_point():
    """Find the first positive branch, returning its location and residuals."""
    # This grid brackets the interior minimum; it does not set its precision.
    messages = np.linspace(0.001, 0.999, 129)
    lambdas = np.array([_cm_lambda_for_message(x) for x in messages])
    minimum = int(np.argmin(lambdas))
    if minimum in (0, len(messages) - 1):
        raise RuntimeError("The critical-point minimum was not bracketed.")

    def tangent_residual(x):
        lam = _cm_lambda_for_message(x)
        return cm_message_derivative(x, lam) - 1

    x = _cm_bisect(tangent_residual, messages[minimum - 1], messages[minimum + 1])
    lam = _cm_lambda_for_message(x)
    return {
        "lambda": lam,
        "x": x,
        "rho": cm_density(x, lam),
        "residual": abs(cm_message(x, lam) - x),
        "tangent_residual": abs(cm_message_derivative(x, lam) - 1),
    }

## Select the largest root and calculate the predicted density

Below $\lambda_c$, the prediction is zero. Above it, the largest root lies
between $x_c$ and one. This gives a reliable bracket even immediately above
the transition, when the two positive roots are too close for an ordinary
grid search to distinguish.

Bisection also avoids the slow convergence of repeated message updates near
a tangency. The function reports its fixed-point residual so the numerical
answer can be checked separately from the simulation.

At exactly $\lambda_c$, we return $(x_c,\rho_c)$ as the **right-hand branch
limit**. This is a plotting convention for the limiting curve. It is not a
claim that a finite graph at that parameter has this deterministic core
fraction. Notebook 07 draws the jump explicitly and treats the critical
window as a finite-size question.

For repeated predictions, calculate the critical point once and pass it in:
```python
critical = cm_critical_point()
predictions = [cm_prediction(lam, critical) for lam in lambdas]
```

In [ ]:
def cm_prediction(lam, critical=None):
    """Largest-root prediction; the critical value denotes its right-hand limit."""
    _, lam = _cm_inputs(0.0, lam)
    if critical is None:
        critical = cm_critical_point()
    if lam < critical["lambda"]:
        return {"lambda": lam, "x": 0.0, "rho": 0.0, "residual": 0.0}

    # Differences of a few floating-point units cannot resolve the double root.
    if lam - critical["lambda"] <= 8 * np.spacing(critical["lambda"]):
        x = critical["x"]
    else:
        x = _cm_bisect(
            lambda value: cm_message(value, lam) - value,
            critical["x"],
            1.0,
        )
    return {
        "lambda": lam,
        "x": x,
        "rho": cm_density(x, lam),
        "residual": abs(cm_message(x, lam) - x),
    }

## From a turning point to a square-root increase in density

The fixed-point equation tells us where the positive branch exists. We now ask
how quickly its predicted core density rises just above the critical parameter.
This coefficient will later enter a finite-size approximation, so it must come
from the deterministic equations rather than a fit to simulated core sizes.

Write $G(x,\lambda)=F_\lambda(x)-x$. At the critical point,
$G(x_c,\lambda_c)=G_x(x_c,\lambda_c)=0$. Set
$\delta=\lambda-\lambda_c$ and $\eta=x-x_c$. Taylor expansion gives

$$
0=G_\lambda\delta+\frac{1}{2}G_{xx}\eta^2
  +O\!\left(|\eta|^3+|\eta|\delta+\delta^2\right),
$$

where derivatives without arguments are evaluated at $(x_c,\lambda_c)$.
If $G_\lambda>0$ and $G_{xx}<0$, the larger root therefore satisfies

$$
x(\lambda)=x_c+b\sqrt{\delta}+O(\delta),
\qquad b=\sqrt{-\frac{2G_\lambda}{G_{xx}}}>0.
$$

The smaller root has the opposite leading sign. The plus sign selects the
largest root, which is the branch reached by pruning from full initial support.
The derivatives of $G$ needed here are $G_\lambda=\partial_\lambda F$ and
$G_{xx}=\partial_x^2 F$, because the subtracted $x$ has neither of these
derivatives.

The density depends on both $x$ and $\lambda$. Expanding that map gives

$$
\rho_\lambda(x(\lambda))
=\rho_c+\rho_x b\sqrt{\delta}+O(\delta),
\qquad
\boxed{a_{\mathrm{CM}}=\rho_x
              \sqrt{-\frac{2G_\lambda}{G_{xx}}}}.
$$

The direct change in $\lambda$ contributes at order $\delta$, so it does not
enter the leading square-root coefficient. A nonzero $\rho_x$ is needed to
transfer the message-coordinate square root to the density.

Here are explicit derivatives that can be implemented without a numerical
derivative inside the coefficient calculation. Define
$u=1-x$ and $E=\exp[-\lambda x(2-x)]$. Differentiating the support equation gives

$$
\begin{aligned}
G_\lambda
 &=x^2E\left[3-2x+2\lambda(2-x)u^2\right],\\
G_{xx}
 &=2\lambda E\left[(1-2x)(3+2\lambda u^2)
                -2\lambda xu^2(5+2\lambda u^2)\right].
\end{aligned}
$$

For the density, differentiate the generating-function expression. The lower
derivative terms cancel:

$$
\rho_x=\frac{x^2}{2}G_0'''(1-x)
      =2\lambda^2x^2uE(3+2\lambda u^2)
      =\lambda x F_\lambda'(x).
$$

Consequently $\rho_x=\lambda_c x_c$ at the fold, an additional identity against
which to check the calculation. The numerical values are approximately

$$
\begin{aligned}
G_\lambda&=0.27011508, & G_{xx}&=-1.70793897,\\
\rho_x&=0.56535072, & b&=0.56241026,\\
a_{\mathrm{CM}}&=0.31795905.
\end{aligned}
$$

These nonzero derivatives verify a nondegenerate local fold numerically.
They do not turn the critical right-hand density into a deterministic prediction
for a finite graph. Nor does this deterministic expansion establish a finite-size
noise scale or distribution; those require separate probabilistic assumptions.

In [ ]:
def cm_message_lambda_derivative(x, lam):
    """Partial derivative of the support map with respect to lambda."""
    x, lam = _cm_inputs(x, lam)
    u = 1 - x
    exponential = np.exp(-lam * x * (2 - x))
    result = x**2 * exponential * (3 - 2 * x + 2 * lam * (2 - x) * u**2)
    return float(result) if result.ndim == 0 else result


def cm_message_second_derivative(x, lam):
    """Second partial derivative of the support map with respect to x."""
    x, lam = _cm_inputs(x, lam)
    u = 1 - x
    exponential = np.exp(-lam * x * (2 - x))
    result = (
        2
        * lam
        * exponential
        * (
            (1 - 2 * x) * (3 + 2 * lam * u**2)
            - 2 * lam * x * u**2 * (5 + 2 * lam * u**2)
        )
    )
    return float(result) if result.ndim == 0 else result


def cm_density_derivative(x, lam):
    """Partial derivative of the predicted density with respect to x."""
    x, lam = _cm_inputs(x, lam)
    u = 1 - x
    result = 2 * lam**2 * x**2 * u * np.exp(-lam * x * (2 - x)) * (3 + 2 * lam * u**2)
    return float(result) if result.ndim == 0 else result


def cm_fold_expansion(critical=None):
    """Return the critical point and theory-derived square-root coefficients."""
    if critical is None:
        critical = cm_critical_point()
    x, lam = critical["x"], critical["lambda"]
    if max(abs(cm_message(x, lam) - x), abs(cm_message_derivative(x, lam) - 1)) > 1e-8:
        raise ValueError(
            "Pass the computed CM critical point, not a regular fixed point."
        )
    parameter_derivative = cm_message_lambda_derivative(x, lam)
    second_derivative = cm_message_second_derivative(x, lam)
    density_derivative = cm_density_derivative(x, lam)
    if not (
        parameter_derivative > 0 and second_derivative < 0 and density_derivative > 0
    ):
        raise RuntimeError(
            "The CM point does not have the required nondegenerate fold."
        )
    message_amplitude = float(np.sqrt(-2 * parameter_derivative / second_derivative))
    return {
        **critical,
        "G_lambda": parameter_derivative,
        "G_xx": second_derivative,
        "rho_x": density_derivative,
        "message_amplitude": message_amplitude,
        "a": density_derivative * message_amplitude,
    }

## Separate matching noise from random-degree noise at the CM fold

The deterministic fold specifies how a small parameter change changes the
predicted density. A probability theorem is needed to specify the random change
caused by sampling a finite graph.
[Janson and Łuczak, Asymptotic normality of the k-core in random graphs,
Theorem 3.5, equation (5.47), and the random-degree conditioning argument in Section 8](https://arxiv.org/pdf/math/0612827)
give the critical law conditional on suitable prescribed degree sequences.
Our benchmark also samples those degrees, independently, so their fluctuations
supply a second contribution.

Let $p=x_c$, $\mu=2\lambda_c$, and conditionally on $K$ let
$Y\sim\operatorname{Binomial}(K,p)$. Define
$g(Y)=Y\mathbf 1_{\{Y\ge3\}}$ and $h_K=\mathbb E[g(Y)\mid K]$.
For the independent twice-Poisson degree ensemble, the two Gaussian variance
contributions at the fold are

$$
\begin{aligned}
v_{\mathrm{match}}
 &=\mathbb E\!\left[\operatorname{Var}(g(Y)\mid K)\right]
        -2\mu p^2(1-p^2),\\
v_{\mathrm{degree}}
 &=\operatorname{Var}(Kp^2-h_K),\\
v_{\mathrm{total}}&=v_{\mathrm{match}}+v_{\mathrm{degree}}.
\end{aligned}
$$

For the first line, let $Z_W$ and $Z_H$ be the Gaussian fluctuations of the
process's active edge-end count and of the ends at vertices having at least
three active ends, respectively. The paper's covariance formulas give

$$
\operatorname{Var}(Z_W)=2\mu(p^2-p^4),\qquad
\operatorname{Var}(Z_H)
=\mathbb E[\operatorname{Var}(g(Y)\mid K)],
$$

and $\operatorname{Cov}(Z_W,Z_H)=p(1-p^2)h'(p)$.
At the fold $h'(p)=2\mu p$, so this covariance equals
$\operatorname{Var}(Z_W)$. The light-end residual is $Z_W-Z_H$; its variance
therefore reduces to the first line. Using only the independent-bin thinning variance would omit its covariance
with the total active-end count. For the second line,
the empirical function is
$\ell_n(p)=n^{-1}\sum_i[K_i p^2-h_{K_i}]$.
Its independent summands have mean zero at criticality, so their ordinary
central limit theorem gives the stated variance. Conditioning on a degree
sequence and then including its empirical fluctuations gives the sum above.

The parameter derivative converts this residual variance to a shift in
$\lambda$:

$$
c=-\partial_\lambda\ell(x_c,\lambda_c)
  =\mu p\,\partial_\lambda F_{\lambda_c}(p)>0,
\qquad
\tau_{\mathrm{CM}}=\frac{\sqrt{v_{\mathrm{total}}}}{c}.
$$

Write $\Phi$ for the standard normal distribution function.
At a critical coordinate $u=\sqrt n(\lambda-\lambda_c)$ the CM limiting
description has occurrence probability $\Phi(u/\tau_{\mathrm{CM}})$ and
conditional scaled density

$$
n^{1/4}(R_n-\rho_c)\mid\{\hbox{substantial core}\}
\ \Longrightarrow\
a_{\mathrm{CM}}\sqrt{u+\tau_{\mathrm{CM}}Z}
       \mid\{u+\tau_{\mathrm{CM}}Z>0\},
\qquad Z\sim N(0,1),
$$

after checking the degree regularity and nondegenerate-minimum hypotheses.
Here the substantial-core cutoff is fixed strictly between zero and $\rho_c$.
The degree law has every exponential moment:
$\mathbb E[A^K]=\exp\{\lambda(A^2-1)\}<\infty$ for each finite $A>1$.
The unique nondegenerate critical minimum was verified above.
Theorem 3.5 covers prescribed-degree multigraphs; applying the random-degree
conditioning argument of Section 8 gives the ensemble used here (the formal
statement of its Theorem 8.3 is for random simple graphs). In contrast, erasing is
a different finite-graph operation and is retained as a separate control.
No corresponding triangle-core fluctuation theorem follows from this calculation.
Nor does this statement automatically describe finite shared-vector controls.

The next function evaluates the expectations by a Poisson probability
recurrence, retaining membership values from zero through a specified maximum.
For each degree $k\ge3$ it uses the exact truncated-binomial moments

$$
\begin{aligned}
h_k&=kp-\mathbb P(Y=1)-2\mathbb P(Y=2),\\
\mathbb E[g(Y)^2\mid K=k]
 &=kp(1-p)+(kp)^2-\mathbb P(Y=1)-4\mathbb P(Y=2).
\end{aligned}
$$

Both quantities are zero for $k<3$. The returned tail-probability bound makes
the finite Poisson sum explicit. Compare two truncation levels to check the
weighted moments as well:

    constants = cm_fluctuation_constants(max_memberships=40)
    finer = cm_fluctuation_constants(max_memberships=60)
    print(constants)
    print({key: abs(constants[key] - finer[key])
           for key in ("variance_matching", "variance_degree", "tau")})

The values are approximately
$v_{\mathrm{match}}=0.10787900$,
$v_{\mathrm{degree}}=0.24909923$,
$c=0.30541951$, and $\tau_{\mathrm{CM}}=1.95624883$.
These constants apply to the retained CM with independent random degrees,
rather than supplying a fitted triangle noise amplitude.

In [ ]:
def cm_fluctuation_constants(critical=None, max_memberships=40):
    """Evaluate critical noise constants for iid twice-Poisson CM degrees."""
    if (
        isinstance(max_memberships, bool)
        or not isinstance(max_memberships, (int, np.integer))
        or max_memberships < 2
    ):
        raise ValueError("max_memberships must be an integer of at least two.")
    expansion = cm_fold_expansion(critical)
    p, lam = expansion["x"], expansion["lambda"]
    mean_degree = 2 * lam

    weights = np.empty(max_memberships + 1, dtype=float)
    weights[0] = np.exp(-lam)
    for membership in range(1, max_memberships + 1):
        weights[membership] = weights[membership - 1] * lam / membership
    degrees = 2 * np.arange(max_memberships + 1)

    support_mean = np.zeros_like(weights)
    support_second_moment = np.zeros_like(weights)
    for membership in range(2, max_memberships + 1):
        degree = int(degrees[membership])
        one = degree * p * (1 - p) ** (degree - 1)
        two = degree * (degree - 1) * p**2 * (1 - p) ** (degree - 2) / 2
        support_mean[membership] = degree * p - one - 2 * two
        support_second_moment[membership] = (
            degree * p * (1 - p) + (degree * p) ** 2 - one - 4 * two
        )

    conditional_variance = support_second_moment - support_mean**2
    variance_matching = float(
        weights @ conditional_variance - 2 * mean_degree * p**2 * (1 - p**2)
    )
    degree_residual = degrees * p**2 - support_mean
    variance_degree = float(
        weights @ degree_residual**2 - (weights @ degree_residual) ** 2
    )
    variance_total = variance_matching + variance_degree
    if variance_matching <= 0 or variance_degree < 0 or variance_total <= 0:
        raise RuntimeError(
            "Increase the Poisson truncation or check the critical point."
        )
    parameter_slope = mean_degree * p * expansion["G_lambda"]
    next_weight = weights[-1] * lam / (max_memberships + 1)
    tail_bound = next_weight / (1 - lam / (max_memberships + 2))
    return {
        "lambda": lam,
        "x": p,
        "rho": expansion["rho"],
        "a": expansion["a"],
        "variance_matching": variance_matching,
        "variance_degree": variance_degree,
        "variance_total": variance_total,
        "parameter_slope": parameter_slope,
        "tau": float(np.sqrt(variance_total) / parameter_slope),
        "poisson_mass": float(weights.sum()),
        "poisson_tail_bound": float(tail_bound),
        "max_memberships": int(max_memberships),
    }

## Verify the derivatives and compare with direct solutions

Analytic formulas are not enough by themselves: a sign error in a second
derivative can reverse the fold or change its amplitude. The first check below
uses centred differences of the original support and density functions:

$$
\begin{aligned}
\partial_\lambda F
 &\approx\frac{F_{\lambda+h}(x)-F_{\lambda-h}(x)}{2h},\\
\partial_x^2F
 &\approx\frac{F_\lambda(x+h)-2F_\lambda(x)+F_\lambda(x-h)}{h^2},\\
\rho_x
 &\approx\frac{\rho_\lambda(x+h)-\rho_\lambda(x-h)}{2h}.
\end{aligned}
$$

Vary $h$ rather than relying on one arbitrary step. The central-difference
truncation error decreases like $h^2$ until floating-point cancellation becomes
important. In particular, a second difference divides a small subtraction by
$h^2$, so ever smaller steps need not give better answers.

A second check solves the original fixed-point equation at
$\lambda=\lambda_c+\delta$ with the existing largest-root solver. It examines

$$
\frac{x(\lambda_c+\delta)-x_c}{\sqrt{\delta}}\longrightarrow b,
\qquad
\frac{\rho(\lambda_c+\delta)-\rho_c}{\sqrt{\delta}}
   \longrightarrow a_{\mathrm{CM}}.
$$

The ratios should approach the coefficients with an error of order
$\sqrt{\delta}$ before roundoff near the double root takes over.
These solutions use the original equation, not the expansion. Residuals are
reported as well, but a tiny residual near a tangent root does not guarantee
equally tiny error in the root.

Loading this notebook only defines the checks. To run them explicitly:

    critical = cm_critical_point()
    expansion = cm_fold_expansion(critical)
    print({key: expansion[key] for key in ("lambda", "rho", "a")})
    derivative_rows = cm_fold_derivative_checks(critical)
    branch_rows = cm_fold_branch_checks(critical)
    for row in derivative_rows:
        print(row)
    for row in branch_rows:
        print(row)

The branch rows also contain the exact and leading-order densities, so they
can be plotted directly. For example:

    import matplotlib.pyplot as plt

    offsets = np.array([row["offset"] for row in branch_rows])
    increments = np.array([row["rho"] - critical["rho"] for row in branch_rows])
    plt.loglog(offsets, increments, "o-", label="Original fixed-point equation")
    plt.loglog(offsets, expansion["a"] * np.sqrt(offsets), "--",
               label="Leading square-root term")
    plt.xlabel(r"$\lambda-\lambda_c$")
    plt.ylabel(r"$\rho-\rho_c$")
    plt.legend()

This is a deterministic theory check, with no random graphs or data files.
A joint comparison with triangle theory can reuse the returned quantities.

In [ ]:
def cm_fold_derivative_checks(critical=None, steps=None):
    """Compare analytic fold derivatives with differences of the original maps."""
    if critical is None:
        critical = cm_critical_point()
    expansion = cm_fold_expansion(critical)
    x, lam = critical["x"], critical["lambda"]
    if steps is None:
        steps = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]
    steps = np.asarray(steps, dtype=float)
    if (
        steps.ndim != 1
        or not len(steps)
        or np.any(~np.isfinite(steps))
        or np.any(steps <= 0)
        or np.any(steps >= min(x, 1 - x, lam))
    ):
        raise ValueError(
            "Use positive finite difference steps inside both parameter domains."
        )

    rows = []
    for step in steps:
        parameter_difference = (
            cm_message(x, lam + step) - cm_message(x, lam - step)
        ) / (2 * step)
        second_difference = (
            cm_message(x + step, lam)
            - 2 * cm_message(x, lam)
            + cm_message(x - step, lam)
        ) / step**2
        density_difference = (cm_density(x + step, lam) - cm_density(x - step, lam)) / (
            2 * step
        )
        if second_difference < 0 and parameter_difference > 0:
            amplitude_difference = density_difference * np.sqrt(
                -2 * parameter_difference / second_difference
            )
        else:
            amplitude_difference = float("nan")
        rows.append(
            {
                "step": float(step),
                "G_lambda_difference": float(parameter_difference),
                "G_xx_difference": float(second_difference),
                "rho_x_difference": float(density_difference),
                "a_difference": float(amplitude_difference),
                "G_lambda_relative_error": abs(
                    parameter_difference / expansion["G_lambda"] - 1
                ),
                "G_xx_relative_error": abs(second_difference / expansion["G_xx"] - 1),
                "rho_x_relative_error": abs(
                    density_difference / expansion["rho_x"] - 1
                ),
                "a_relative_error": abs(amplitude_difference / expansion["a"] - 1),
            }
        )
    return rows


def cm_fold_branch_checks(critical=None, offsets=None):
    """Compare the exact positive branch with its leading square-root expansion."""
    if critical is None:
        critical = cm_critical_point()
    expansion = cm_fold_expansion(critical)
    if offsets is None:
        offsets = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]
    offsets = np.asarray(offsets, dtype=float)
    if (
        offsets.ndim != 1
        or not len(offsets)
        or np.any(~np.isfinite(offsets))
        or np.any(offsets <= 8 * np.spacing(critical["lambda"]))
    ):
        raise ValueError(
            "Use finite positive offsets resolvable above the double root."
        )

    rows = []
    for offset in offsets:
        prediction = cm_prediction(critical["lambda"] + offset, critical)
        root_offset = float(np.sqrt(offset))
        message_ratio = (prediction["x"] - critical["x"]) / root_offset
        density_ratio = (prediction["rho"] - critical["rho"]) / root_offset
        rows.append(
            {
                "offset": float(offset),
                "lambda": prediction["lambda"],
                "x": prediction["x"],
                "rho": prediction["rho"],
                "rho_approximation": critical["rho"] + expansion["a"] * root_offset,
                "message_ratio": message_ratio,
                "density_ratio": density_ratio,
                "message_ratio_error": abs(
                    message_ratio - expansion["message_amplitude"]
                ),
                "density_ratio_error": abs(density_ratio - expansion["a"]),
                "residual": prediction["residual"],
            }
        )
    return rows

## Checks to carry into the numerical comparison

The formulas can be checked independently by summing over
$D\sim\operatorname{Poisson}(\lambda)$. For $j\ge1$, the contribution to
the message equation is

$$
\frac{j\,\mathbb{P}(D=j)}{\lambda}
  \mathbb{P}(\operatorname{Binomial}(2j-1,x)\ge2),
$$

and the contribution to the density is

$$
\mathbb{P}(D=j)
  \mathbb{P}(\operatorname{Binomial}(2j,x)\ge3).
$$

These sums agree with the generating-function formulas. The automated checks
evaluate both expressions independently, check the two tangency equations,
and examine fixed points just below and above the transition.

Notebook 07 overlays this limiting curve on independently sampled
configuration-model graphs at several sizes. Away from the transition,
agreement should improve as finite-size effects decrease. Near the transition,
some trials may have little core and others a substantial one; their mean
need not resemble the discontinuous limiting curve.

Finally, matching a degree distribution does not match all other structure.
The triangle model has persistent short cycles and dependent neighbour
support, whereas the configuration model pairs half-edges freely. Differences
are evidence about these structural changes together. The scalar equation in
this notebook is not a theoretical prediction for the triangle model.

## From occurrence probabilities to a conditional size prediction

The CM theorem supplies a leading effective parameter scale $\tau$ without
fitting. For triangles the corresponding noise law is a hypothesis. A common
finite-size approximation uses a random distance $V=m+sZ$, with $Z$ standard
normal and $s>0$, and calls $V>0$ survival. If
$m=\sqrt n(\lambda-\mu_n)$ and $s=\sqrt n\,w_n$, its occurrence probability is
$\Phi(m/s)$. Estimate $\mu_n,w_n$ from survival indicators alone.

Once the derivative calculation fixes $a>0$ and $\rho_c$, predict

$$
n^{1/4}(R_n-\rho_c)\mid\text{survival}\ \approx\ Y:=a\sqrt V\mid\{V>0\}.
$$

For $y\geq0$ and $0<q<1$,

$$
F_Y(y)=1-\frac{\Phi((m-(y/a)^2)/s)}{\Phi(m/s)},\qquad
Q_Y(q)=a\sqrt{m-s\Phi^{-1}((1-q)\Phi(m/s))}.
$$

The tail form avoids subtracting two normal distribution functions close to
one. The functions below implement this **specified** law; they do not fit a
model or validate its applicability to triangle graphs. Their arguments
`centre` and `width` are the scaled $m,s$, not $\mu_n,w_n$. The density location,
amplitude, and spread may not be refitted using the conditional sizes.
A predicted second moment is
$a^2[m+s\phi(m/s)/\Phi(m/s)]$; the first moment follows by one-dimensional
quadrature. Chapter 3 gives the assumptions and moment integral.

Run [the deterministic comparison notebook](theory_comparison.ipynb), or
`make theory-figures`, to evaluate both models, inspect verification diagnostics,
and regenerate the paper's table and plot. No graph simulation is needed.

In [ ]:
from math import erfc, isfinite, isnan, sqrt
from statistics import NormalDist


def _stochastic_fold_inputs(amplitude, centre, width):
    """Validate a>0, scaled m, and scaled s>0; return them and P(V>0)."""
    amplitude, centre, width = map(float, (amplitude, centre, width))
    if not all(isfinite(value) for value in (amplitude, centre, width)):
        raise ValueError("Amplitude, centre, and width must be finite.")
    if amplitude <= 0 or width <= 0:
        raise ValueError("Amplitude and width must be positive.")
    survival = 0.5 * erfc(-centre / (width * sqrt(2)))
    if survival == 0:
        raise ValueError(
            "Survival probability underflows; use a resolvable critical-window point."
        )
    return amplitude, centre, width, survival


def stochastic_fold_cdf(value, amplitude, centre, width):
    """CDF of a*sqrt(m+sZ) conditional on m+sZ>0, for scalar value."""
    amplitude, centre, width, survival = _stochastic_fold_inputs(
        amplitude, centre, width
    )
    value = float(value)
    if isnan(value):
        raise ValueError("The evaluation value must not be NaN.")
    if value <= 0:
        return 0.0
    if not isfinite(value):
        return 1.0
    tail = 0.5 * erfc(((value / amplitude) ** 2 - centre) / (width * sqrt(2)))
    return min(1.0, max(0.0, 1 - tail / survival))


def stochastic_fold_quantile(probability, amplitude, centre, width):
    """Quantile of the same conditional law; endpoints give zero and infinity."""
    amplitude, centre, width, survival = _stochastic_fold_inputs(
        amplitude, centre, width
    )
    probability = float(probability)
    if not isfinite(probability) or not 0 <= probability <= 1:
        raise ValueError("The quantile probability must lie in [0, 1].")
    if probability == 0:
        return 0.0
    if probability == 1:
        return float("inf")
    normal_quantile = NormalDist().inv_cdf((1 - probability) * survival)
    return amplitude * sqrt(max(0.0, centre - width * normal_quantile))